In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kumarajarshi/life-expectancy-who/Life Expectancy Data.csv


### Read in Data

In [2]:
data = pd.read_csv('/kaggle/input/datasets/kumarajarshi/life-expectancy-who/Life Expectancy Data.csv')
df = pd.DataFrame(data)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          2938 non-null   object 
 1   Year                             2938 non-null   int64  
 2   Status                           2938 non-null   object 
 3   Life expectancy                  2928 non-null   float64
 4   Adult Mortality                  2928 non-null   float64
 5   infant deaths                    2938 non-null   int64  
 6   Alcohol                          2744 non-null   float64
 7   percentage expenditure           2938 non-null   float64
 8   Hepatitis B                      2385 non-null   float64
 9   Measles                          2938 non-null   int64  
 10   BMI                             2904 non-null   float64
 11  under-five deaths                2938 non-null   int64  
 12  Polio               

### Standardize Country and Status

In [3]:
df['Country'] = df['Country'].astype(str)
df['Status'] = df['Status'].astype(str)

df['Country'] = df['Country'].str.strip().str.capitalize()
df['Status'] = df['Status'].str.strip().str.capitalize()

df.columns = (df.columns
    .str.strip()
    .str.upper()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace(r'[^\w\s]', '', regex=True)
)
print(df.head())


       COUNTRY  YEAR      STATUS  LIFE_EXPECTANCY  ADULT_MORTALITY  \
0  Afghanistan  2015  Developing             65.0            263.0   
1  Afghanistan  2014  Developing             59.9            271.0   
2  Afghanistan  2013  Developing             59.9            268.0   
3  Afghanistan  2012  Developing             59.5            272.0   
4  Afghanistan  2011  Developing             59.2            275.0   

   INFANT_DEATHS  ALCOHOL  PERCENTAGE_EXPENDITURE  HEPATITIS_B  MEASLES  ...  \
0             62     0.01               71.279624         65.0     1154  ...   
1             64     0.01               73.523582         62.0      492  ...   
2             66     0.01               73.219243         64.0      430  ...   
3             69     0.01               78.184215         67.0     2787  ...   
4             71     0.01                7.097109         68.0     3013  ...   

   POLIO  TOTAL_EXPENDITURE  DIPHTHERIA  HIVAIDS         GDP  POPULATION  \
0    6.0              

### Find Duplicates

In [4]:
duplicated = df[df.duplicated()]
print(duplicated)

Empty DataFrame
Columns: [COUNTRY, YEAR, STATUS, LIFE_EXPECTANCY, ADULT_MORTALITY, INFANT_DEATHS, ALCOHOL, PERCENTAGE_EXPENDITURE, HEPATITIS_B, MEASLES, BMI, UNDER_FIVE_DEATHS, POLIO, TOTAL_EXPENDITURE, DIPHTHERIA, HIVAIDS, GDP, POPULATION, THINNESS__1_19_YEARS, THINNESS_5_9_YEARS, INCOME_COMPOSITION_OF_RESOURCES, SCHOOLING]
Index: []

[0 rows x 22 columns]


### Find/Fill Null/NaN

In [5]:
# for cols in df:
#     count = df[cols].isna().sum().sum()
#     print(f"{cols} : {count}")
to_zero_cols = ['ALCOHOL',
                'HEPATITIS_B',
                 'POLIO',
                 'DIPHTHERIA',
                 'SCHOOLING',
                 'TOTAL_EXPENDITURE'
                ]
df[to_zero_cols] = df[to_zero_cols].fillna(0)
df = df.fillna({'LIFE_EXPECTANCY': df['LIFE_EXPECTANCY'].mean(),
                'ADULT_MORTALITY': df['ADULT_MORTALITY'].mean(),
                'BMI': df['BMI'].mean(),
                'GDP' : df['GDP'].mean(),
                'POPULATION' : df['POPULATION'].mean(),
                'INCOME_COMPOSITION_OF_RESOURCES' : df['INCOME_COMPOSITION_OF_RESOURCES'].mean()})
del df['THINNESS_5_9_YEARS']
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   COUNTRY                          2938 non-null   object 
 1   YEAR                             2938 non-null   int64  
 2   STATUS                           2938 non-null   object 
 3   LIFE_EXPECTANCY                  2938 non-null   float64
 4   ADULT_MORTALITY                  2938 non-null   float64
 5   INFANT_DEATHS                    2938 non-null   int64  
 6   ALCOHOL                          2938 non-null   float64
 7   PERCENTAGE_EXPENDITURE           2938 non-null   float64
 8   HEPATITIS_B                      2938 non-null   float64
 9   MEASLES                          2938 non-null   int64  
 10  BMI                              2938 non-null   float64
 11  UNDER_FIVE_DEATHS                2938 non-null   int64  
 12  POLIO               

### Make New Columns

In [6]:
df['DEATHS_FROM_DISEASE'] = df['ALCOHOL'] + df['HEPATITIS_B'] + df['MEASLES'] + df['POLIO'] + df['DIPHTHERIA'] + df['HIVAIDS']
df['CHILD_DEATH_TO_THINNESS'] =  df['UNDER_FIVE_DEATHS'] / df['THINNESS__1_19_YEARS']


print(df.head())



       COUNTRY  YEAR      STATUS  LIFE_EXPECTANCY  ADULT_MORTALITY  \
0  Afghanistan  2015  Developing             65.0            263.0   
1  Afghanistan  2014  Developing             59.9            271.0   
2  Afghanistan  2013  Developing             59.9            268.0   
3  Afghanistan  2012  Developing             59.5            272.0   
4  Afghanistan  2011  Developing             59.2            275.0   

   INFANT_DEATHS  ALCOHOL  PERCENTAGE_EXPENDITURE  HEPATITIS_B  MEASLES  ...  \
0             62     0.01               71.279624         65.0     1154  ...   
1             64     0.01               73.523582         62.0      492  ...   
2             66     0.01               73.219243         64.0      430  ...   
3             69     0.01               78.184215         67.0     2787  ...   
4             71     0.01                7.097109         68.0     3013  ...   

   TOTAL_EXPENDITURE  DIPHTHERIA  HIVAIDS         GDP  POPULATION  \
0               8.16        6